# Weapon Detection V2 — Crash-Proof Google Colab Training & Benchmarking Suite

This notebook keeps **all heavy datasets and training checkpoints off your laptop** AND **automatically saves & resumes progress on Google Drive** so a Colab timeout never makes you start over:
- **Zero local laptop storage:** Raw datasets live only in Colab `/content/work/`.
- **Automatic dataset crop cache:** Once Cell A2/A2b prepares crops, they cache cleanly to `MyDrive/weapon_v2_runs/verifier_crops_cache.zip`. On reconnect, Cell A2/A2b restores all crops in seconds without re-downloading.
- **Per-epoch atomic auto-resume:**
  - **Stage-2 Verifier (`Cell A3`):** Locks the train/val split, atomically saves `crop_verifier_convnext_tiny.pt` whenever validation AUC improves, and saves full resume state (`verifier_resume_state.pt`) after **every epoch**.
  - **Stage-1 `YOLO11x` & `RT-DETR-L` (`Cell B2`/`B3`):** Save to Drive **every epoch** (`save_period=1`) and auto-resume from `last.pt` on re-run without flipping any flags.

### Recommended Order
1. **Run Section 1 (Setup)** to verify GPU and mount Google Drive.
2. **Run Pipeline A (~20 min on Free T4)** to train **`crop_verifier_convnext_tiny.pt`**. *(Cell A1 local webcam mining is optional — Cell A2b automatically pulls public COCO 2017 phones, remotes, umbrellas, table rims, and backpacks inside Colab.)*
3. **Run Pipeline B (T4 w/ Drive Auto-Resume or Pro L4/A100)** to fine-tune multi-class **`YOLO11x`** + explicit distractors and benchmark zero-NMS **`RT-DETR-L`** on stacked weapons.
4. **Run Section 4** to measure **FAPH (False Alarms Per Hour)** and **Incident Recall / TTFD** on your Golden Stress-Test clips.


## 1. Setup: Verify GPU, Mount Google Drive & Load Resume Helpers


In [ ]:
import os
import zipfile
from pathlib import Path
import torch

!nvidia-smi
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount("/content/drive")

DRIVE_RUNS = Path("/content/drive/MyDrive/weapon_v2_runs")
DRIVE_RUNS.mkdir(parents=True, exist_ok=True)

WORK_DIR = Path("/content/work")
WORK_DIR.mkdir(parents=True, exist_ok=True)
VERIFIER_RAW_DIR = WORK_DIR / "verifier_crops_raw"
VERIFIER_RAW_DIR.mkdir(parents=True, exist_ok=True)
DL_DIR = WORK_DIR / "downloads"
DL_DIR.mkdir(parents=True, exist_ok=True)

CROPS_CACHE_ZIP = DRIVE_RUNS / "verifier_crops_cache.zip"
FORCE_REBUILD_CROPS = False  # Set True only if you want to wipe & rebuild cached crops

def atomic_torch_save(obj, dst_path: Path) -> None:
    """Write checkpoint to a temporary file then atomically rename so Drive files never corrupt."""
    dst = Path(dst_path)
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_suffix(dst.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, dst)

def restore_crops_cache_from_drive() -> bool:
    if FORCE_REBUILD_CROPS or not CROPS_CACHE_ZIP.is_file():
        return False
    print(f"[Cache Hit] Restoring crops from {CROPS_CACHE_ZIP}...")
    with zipfile.ZipFile(CROPS_CACHE_ZIP, "r") as zf:
        zf.extractall(VERIFIER_RAW_DIR)
    counts = {d.name: len(list(d.glob("*.jpg"))) for d in sorted(VERIFIER_RAW_DIR.iterdir()) if d.is_dir()}
    print("[Cache Restored] Class summary:", counts)
    return True

def save_crops_cache_to_drive() -> None:
    tmp_zip = CROPS_CACHE_ZIP.with_suffix(".zip.tmp")
    with zipfile.ZipFile(tmp_zip, "w", compression=zipfile.ZIP_STORED) as zf:
        for p in sorted(VERIFIER_RAW_DIR.rglob("*")):
            if p.is_file():
                zf.write(p, arcname=p.relative_to(VERIFIER_RAW_DIR).as_posix())
    os.replace(tmp_zip, CROPS_CACHE_ZIP)
    size_mb = CROPS_CACHE_ZIP.stat().st_size / (1024 * 1024)
    print(f"[Cache Saved] {CROPS_CACHE_ZIP} ({size_mb:.1f} MB)")

print("Drive checkpoint dir:", DRIVE_RUNS)
print("Ephemeral work dir:  ", WORK_DIR)

!pip install -q ultralytics>=8.3.0 albumentations>=1.4.0 roboflow scikit-learn


## 2. Pipeline A (Highest ROI, ~20 Min on Free T4): Train Stage-2 `ConvNeXt-Tiny` Crop Verifier

Trains a Stage-2 classifier on **full-resolution padded crops (`pad_ratio=0.30`)** for every uncertain Stage-1 box (`conf < 0.85`):
- **Positive class:** `weapon`
- **Hard distractor classes:** `phone`, `umbrella`, `table_bg`, `other_bg`
- **Progress safety:** Crops cache to `verifier_crops_cache.zip` on Drive, and training saves/resumes **every epoch**.


In [ ]:
# Cell A1 (Optional): Load Locally Harvested Hard-Negative ZIP (from tools/mine_hard_negatives.py)
# Safe to cancel upload or skip — Cell A2b automatically adds public COCO 2017 distractors.
import zipfile
from pathlib import Path

ZIP_FROM_DRIVE = str(DRIVE_RUNS / "hard_negatives_upload.zip")

if Path(ZIP_FROM_DRIVE).is_file():
    zip_path = Path(ZIP_FROM_DRIVE)
    print("Found hard-negative bundle on Drive:", zip_path)
else:
    from google.colab import files
    print("Optional: upload 'hard_negatives_upload.zip' (or click Cancel upload to skip):")
    uploaded = files.upload()
    zip_path = Path(next(iter(uploaded.keys()))) if uploaded else None

if zip_path and zip_path.is_file():
    with zipfile.ZipFile(zip_path, "r") as zf:
        for member in zf.infolist():
            if member.filename.lower().endswith((".jpg", ".jpeg", ".png")):
                parts = Path(member.filename).parts
                cls_name = parts[0] if len(parts) > 1 else "other_bg"
                dst_dir = VERIFIER_RAW_DIR / cls_name
                dst_dir.mkdir(parents=True, exist_ok=True)
                (dst_dir / Path(member.filename).name).write_bytes(zf.read(member))
    counts = {d.name: len(list(d.glob("*.jpg"))) for d in VERIFIER_RAW_DIR.iterdir() if d.is_dir()}
    print("Extracted mined distractor crops:", counts)
else:
    print("No local ZIP uploaded; proceeding with public Roboflow + COCO datasets.")


In [ ]:
# Cell A2: Download Weapon + Public Distractor Crops into /content/work (Auto-Cached on Drive)
import getpass
import random
import cv2
import yaml
from pathlib import Path

ROBOFLOW_DONE = VERIFIER_RAW_DIR / ".roboflow_done"

if not FORCE_REBUILD_CROPS and not ROBOFLOW_DONE.is_file():
    restore_crops_cache_from_drive()

if ROBOFLOW_DONE.is_file() and not FORCE_REBUILD_CROPS:
    print("[Skip] Roboflow crops already present from cache.")
else:
    from roboflow import Roboflow
    ROBOFLOW_KEY = os.environ.get("ROBOFLOW_API_KEY") or getpass.getpass("Enter Roboflow API key: ")
    rf = Roboflow(api_key=ROBOFLOW_KEY)

    weapon_loc = DL_DIR / "weapon_v5"
    phone_loc = DL_DIR / "phone_v3"
    umbrella_loc = DL_DIR / "umbrella_v1"

    if not (weapon_loc / "data.yaml").is_file():
        rf.workspace("yolov7test-u13vc").project("weapon-detection-m7qso").version(5).download("yolov8", location=str(weapon_loc))
    if not (phone_loc / "data.yaml").is_file():
        rf.workspace("fyp-kbmf2").project("phone-finder-2.0-fbxkx").version(3).download("yolov8", location=str(phone_loc))
    if not (umbrella_loc / "data.yaml").is_file():
        rf.workspace("objectdetection-gd0bu").project("umbrella-6hxa2").version(1).download("yolov8", location=str(umbrella_loc))

    def extract_padded_rgb_from_yolo(ds_root: Path, out_cls: str, max_samples: int = 2500, pad_ratio: float = 0.30, target_names=None):
        out_dir = VERIFIER_RAW_DIR / out_cls
        out_dir.mkdir(parents=True, exist_ok=True)
        existing = len(list(out_dir.glob("*.jpg")))
        if existing >= max_samples:
            print(f"[{out_cls}] Already has {existing} crops.")
            return

        keep_ids = None
        yaml_file = ds_root / "data.yaml"
        if target_names is not None and yaml_file.is_file():
            meta = yaml.safe_load(yaml_file.read_text())
            names = meta.get("names", [])
            if isinstance(names, dict):
                keep_ids = {int(k) for k, v in names.items() if any(t in str(v).lower() for t in target_names)}
            else:
                keep_ids = {int(i) for i, v in enumerate(names) if any(t in str(v).lower() for t in target_names)}

        img_paths = list(ds_root.rglob("images/*.jpg")) + list(ds_root.rglob("images/*.png"))
        random.shuffle(img_paths)
        saved = existing
        for img_p in img_paths:
            if saved >= max_samples:
                break
            lbl_p = img_p.parent.parent / "labels" / f"{img_p.stem}.txt"
            if not lbl_p.is_file():
                continue
            frame = cv2.imread(str(img_p))
            if frame is None:
                continue
            fh, fw = frame.shape[:2]
            for b_idx, line in enumerate(lbl_p.read_text().strip().splitlines()):
                parts = line.split()
                if len(parts) < 5:
                    continue
                cid = int(float(parts[0]))
                if keep_ids is not None and cid not in keep_ids:
                    continue
                xc, yc, bw, bh = map(float, parts[1:5])
                x1 = int((xc - bw * 0.5) * fw)
                y1 = int((yc - bh * 0.5) * fh)
                x2 = int((xc + bw * 0.5) * fw)
                y2 = int((yc + bh * 0.5) * fh)
                box_w, box_h = max(1, x2 - x1), max(1, y2 - y1)
                if box_w < 14 or box_h < 14:
                    continue
                px1 = max(0, x1 - int(round(box_w * pad_ratio)))
                py1 = max(0, y1 - int(round(box_h * pad_ratio)))
                px2 = min(fw, x2 + int(round(box_w * pad_ratio)))
                py2 = min(fh, y2 + int(round(box_h * pad_ratio)))
                crop = cv2.resize(frame[py1:py2, px1:px2], (256, 256), interpolation=cv2.INTER_AREA)
                cv2.imwrite(str(out_dir / f"{out_cls}_{img_p.stem}_{b_idx}.jpg"), crop, [cv2.IMWRITE_JPEG_QUALITY, 90])
                saved += 1
                if saved >= max_samples:
                    break
        print(f"[{out_cls}] Total crops ready: {saved}")

    extract_padded_rgb_from_yolo(
        weapon_loc, "weapon", max_samples=3500,
        target_names={"pistol", "knife", "weapon", "gun", "rifle", "shotgun", "larga", "curta", "arma"}
    )
    extract_padded_rgb_from_yolo(phone_loc, "phone", max_samples=2000)
    extract_padded_rgb_from_yolo(umbrella_loc, "umbrella", max_samples=1500)

    bg_dir = VERIFIER_RAW_DIR / "other_bg"
    bg_dir.mkdir(parents=True, exist_ok=True)
    empty_imgs = [p for p in weapon_loc.rglob("images/*.jpg") if (p.parent.parent / "labels" / f"{p.stem}.txt").is_file() and (p.parent.parent / "labels" / f"{p.stem}.txt").read_text().strip() == ""]
    for idx, p in enumerate(empty_imgs[:600]):
        im = cv2.imread(str(p))
        if im is None:
            continue
        h, w = im.shape[:2]
        cw, ch = max(48, w // 4), max(48, h // 4)
        rx = random.randint(0, max(0, w - cw))
        ry = random.randint(0, max(0, h - ch))
        crop = cv2.resize(im[ry:ry+ch, rx:rx+cw], (256, 256))
        cv2.imwrite(str(bg_dir / f"bg_{idx:04d}.jpg"), crop)

    ROBOFLOW_DONE.write_text("ok")
    save_crops_cache_to_drive()

print("Stage-2 class summary after A2:", {d.name: len(list(d.glob("*.jpg"))) for d in sorted(VERIFIER_RAW_DIR.iterdir()) if d.is_dir()})


In [ ]:
# Cell A2b: Zero-Effort Public COCO 2017 Distractor Crops (Auto-Cached to Drive)
# Downloads COCO val2017 into Colab ephemeral storage and extracts
# phones, remotes, umbrellas, table rims, backpacks, handbags, and hair driers.
import json
import random
import urllib.request
import zipfile
from collections import defaultdict
from pathlib import Path
import cv2

COCO_DONE = VERIFIER_RAW_DIR / ".coco_done"

if not FORCE_REBUILD_CROPS and not COCO_DONE.is_file():
    restore_crops_cache_from_drive()

if COCO_DONE.is_file() and not FORCE_REBUILD_CROPS:
    print("[Skip] COCO 2017 distractor crops already present from cache.")
else:
    COCO_DIR = WORK_DIR / "coco"
    COCO_DIR.mkdir(parents=True, exist_ok=True)
    VAL_IMG_DIR = COCO_DIR / "val2017"
    ANN_JSON = COCO_DIR / "annotations" / "instances_val2017.json"

    if not VAL_IMG_DIR.is_dir():
        val_zip = COCO_DIR / "val2017.zip"
        print("Downloading COCO val2017 images (~778 MB into Colab ephemeral storage)...")
        urllib.request.urlretrieve("http://images.cocodataset.org/zips/val2017.zip", val_zip)
        with zipfile.ZipFile(val_zip, "r") as zf:
            zf.extractall(COCO_DIR)
        val_zip.unlink(missing_ok=True)

    if not ANN_JSON.is_file():
        ann_zip = COCO_DIR / "annotations_trainval2017.zip"
        print("Downloading COCO 2017 annotations (~241 MB into Colab ephemeral storage)...")
        urllib.request.urlretrieve("http://images.cocodataset.org/annotations/annotations_trainval2017.zip", ann_zip)
        with zipfile.ZipFile(ann_zip, "r") as zf:
            zf.extract("annotations/instances_val2017.json", COCO_DIR)
        ann_zip.unlink(missing_ok=True)

    coco = json.loads(ANN_JSON.read_text(encoding="utf-8"))
    name_to_id = {c["name"]: c["id"] for c in coco["categories"]}
    id_to_img = {im["id"]: im["file_name"] for im in coco["images"]}

    TARGET_MAP = {
        "cell phone": "phone",
        "remote": "phone",
        "umbrella": "umbrella",
        "dining table": "table_bg",
        "backpack": "other_bg",
        "handbag": "other_bg",
        "hair drier": "other_bg",
    }
    cat_id_to_bucket = {name_to_id[n]: b for n, b in TARGET_MAP.items() if n in name_to_id}
    BUCKET_LIMITS = {"phone": 600, "umbrella": 400, "table_bg": 500, "other_bg": 500}

    grouped_anns = defaultdict(list)
    for ann in coco["annotations"]:
        if ann.get("iscrowd", 0):
            continue
        bucket = cat_id_to_bucket.get(ann["category_id"])
        if bucket is not None:
            grouped_anns[bucket].append(ann)

    PAD_RATIO = 0.30

    def save_padded_coco_crop(frame, box_xyxy, dst_path: Path, pad_ratio: float = PAD_RATIO) -> bool:
        fh, fw = frame.shape[:2]
        x1, y1, x2, y2 = box_xyxy
        bw, bh = max(1, x2 - x1), max(1, y2 - y1)
        if bw < 14 or bh < 14:
            return False
        px1 = max(0, x1 - int(round(bw * pad_ratio)))
        py1 = max(0, y1 - int(round(bh * pad_ratio)))
        px2 = min(fw, x2 + int(round(bw * pad_ratio)))
        py2 = min(fh, y2 + int(round(bh * pad_ratio)))
        if px2 <= px1 or py2 <= py1:
            return False
        crop = cv2.resize(frame[py1:py2, px1:px2], (256, 256), interpolation=cv2.INTER_AREA)
        return bool(cv2.imwrite(str(dst_path), crop, [cv2.IMWRITE_JPEG_QUALITY, 90]))

    for bucket, limit in BUCKET_LIMITS.items():
        out_dir = VERIFIER_RAW_DIR / bucket
        out_dir.mkdir(parents=True, exist_ok=True)
        anns = list(grouped_anns.get(bucket, []))
        random.shuffle(anns)
        added = 0
        for idx, ann in enumerate(anns):
            if added >= limit:
                break
            fname = id_to_img.get(ann["image_id"])
            if not fname:
                continue
            frame = cv2.imread(str(VAL_IMG_DIR / fname))
            if frame is None:
                continue
            fh, fw = frame.shape[:2]
            bx, by, bw, bh = ann["bbox"]
            x1, y1 = int(round(bx)), int(round(by))
            x2, y2 = int(round(bx + bw)), int(round(by + bh))

            if bucket == "table_bg":
                if save_padded_coco_crop(frame, (x1, y1, x2, y2), out_dir / f"coco_tbl_{idx}.jpg", pad_ratio=0.15):
                    added += 1
                if added < limit and bw >= 40 and bh >= 40:
                    rim_h = max(18, int(round(bh * 0.16)))
                    if save_padded_coco_crop(frame, (x1, max(0, y1 - rim_h), x2, min(fh, y1 + rim_h)), out_dir / f"coco_rim_t_{idx}.jpg", pad_ratio=0.10):
                        added += 1
                if added < limit and bw >= 40 and bh >= 40:
                    rim_h = max(18, int(round(bh * 0.16)))
                    if save_padded_coco_crop(frame, (x1, max(0, y2 - rim_h), x2, min(fh, y2 + rim_h)), out_dir / f"coco_rim_b_{idx}.jpg", pad_ratio=0.10):
                        added += 1
            else:
                if save_padded_coco_crop(frame, (x1, y1, x2, y2), out_dir / f"coco_{bucket}_{idx}.jpg", pad_ratio=PAD_RATIO):
                    added += 1
        print(f"[COCO val2017 -> {bucket}] Added {added} crops")

    COCO_DONE.write_text("ok")
    save_crops_cache_to_drive()

print("Updated Stage-2 class summary:", {d.name: len(list(d.glob("*.jpg"))) for d in sorted(VERIFIER_RAW_DIR.iterdir()) if d.is_dir()})


In [ ]:
# Cell A3: Train Stage-2 ConvNeXt-Tiny Verifier (Per-Epoch Auto-Resume & Atomic Drive Save)
import copy
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as tv_models
import albumentations as A
from sklearn.metrics import classification_report, roc_auc_score
import cv2
from pathlib import Path

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

IMG_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 15
LR = 3e-4
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

OUT_CKPT = DRIVE_RUNS / "crop_verifier_convnext_tiny.pt"
RESUME_CKPT = DRIVE_RUNS / "verifier_resume_state.pt"
FORCE_RESTART_VERIFIER = False  # Set True only to restart training from Epoch 1

# Auto-restore crops if reconnecting directly to Cell A3
if not any(VERIFIER_RAW_DIR.glob("*/*.jpg")):
    restore_crops_cache_from_drive()

resume_data = None
if RESUME_CKPT.is_file() and not FORCE_RESTART_VERIFIER:
    resume_data = torch.load(RESUME_CKPT, map_location="cpu", weights_only=False)
    classes = list(resume_data["classes"])
    cls_to_idx = {c: i for i, c in enumerate(classes)}
    train_rel = list(resume_data["train_rel"])
    val_rel = list(resume_data["val_rel"])
    train_items = [(VERIFIER_RAW_DIR / rel, y) for rel, y in train_rel if (VERIFIER_RAW_DIR / rel).is_file()]
    val_items = [(VERIFIER_RAW_DIR / rel, y) for rel, y in val_rel if (VERIFIER_RAW_DIR / rel).is_file()]
    assert len(train_items) > 0 and len(val_items) > 0, "Resume split paths missing from /content/work. Run Cell A2/A2b first."
    print(f"[Resume Found] Last completed epoch={resume_data['epoch']}/{EPOCHS} | best_auc={resume_data['best_auc']:.4f}")
else:
    classes = sorted([d.name for d in VERIFIER_RAW_DIR.iterdir() if d.is_dir() and len(list(d.glob("*.jpg"))) >= 10])
    assert "weapon" in classes, f"Expected 'weapon' folder in {VERIFIER_RAW_DIR}, got {classes}"
    cls_to_idx = {c: i for i, c in enumerate(classes)}
    train_rel, val_rel = [], []
    for c in classes:
        files = sorted((VERIFIER_RAW_DIR / c).glob("*.jpg"))
        random.shuffle(files)
        split = max(1, int(0.85 * len(files)))
        train_rel += [(p.relative_to(VERIFIER_RAW_DIR).as_posix(), cls_to_idx[c]) for p in files[:split]]
        val_rel += [(p.relative_to(VERIFIER_RAW_DIR).as_posix(), cls_to_idx[c]) for p in files[split:]]
    train_items = [(VERIFIER_RAW_DIR / rel, y) for rel, y in train_rel]
    val_items = [(VERIFIER_RAW_DIR / rel, y) for rel, y in val_rel]

print("Training Stage-2 classes:", classes, f"(train={len(train_items)}, val={len(val_items)})")

train_aug = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.Perspective(scale=(0.03, 0.08), p=0.4),
    A.Rotate(limit=25, border_mode=cv2.BORDER_REFLECT_101, p=0.5),
    A.OneOf([
        A.MotionBlur(blur_limit=(3, 7), p=1.0),
        A.GaussianBlur(blur_limit=(3, 5), p=1.0),
    ], p=0.35),
    A.ImageCompression(quality_range=(35, 90), p=0.45),
    A.ISONoise(p=0.30),
    A.RandomBrightnessContrast(brightness_limit=0.30, contrast_limit=0.25, p=0.5),
    A.ToGray(p=0.15),
])
val_aug = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE)])

class CropDataset(Dataset):
    def __init__(self, items, aug):
        self.items = items
        self.aug = aug
        self.mean = np.array(MEAN, dtype=np.float32).reshape(3, 1, 1)
        self.std = np.array(STD, dtype=np.float32).reshape(3, 1, 1)

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        p, y = self.items[idx]
        bgr = cv2.imread(str(p))
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        img = self.aug(image=rgb)["image"].astype(np.float32) / 255.0
        chw = (img.transpose(2, 0, 1) - self.mean) / self.std
        return torch.from_numpy(chw), torch.tensor(y, dtype=torch.long)

train_loader = DataLoader(CropDataset(train_items, train_aug), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(CropDataset(val_items, val_aug), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

counts = np.bincount([y for _, y in train_items], minlength=len(classes)).astype(np.float32)
weights = torch.tensor(counts.sum() / (len(classes) * np.maximum(counts, 1.0)), dtype=torch.float32).cuda()

model = tv_models.convnext_tiny(weights="DEFAULT")
model.classifier[2] = nn.Linear(model.classifier[2].in_features, len(classes))
model = model.cuda()

criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler("cuda")

start_epoch = 1
best_auc = -1.0
best_state = None
weapon_idx = cls_to_idx["weapon"]

if resume_data is not None:
    model.load_state_dict(resume_data["model_state"])
    optimizer.load_state_dict(resume_data["optimizer_state"])
    scheduler.load_state_dict(resume_data["scheduler_state"])
    scaler.load_state_dict(resume_data["scaler_state"])
    start_epoch = int(resume_data["epoch"]) + 1
    best_auc = float(resume_data["best_auc"])
    best_state = resume_data["best_state"]

if start_epoch > EPOCHS:
    print(f"[Complete] Stage-2 verifier already finished all {EPOCHS} epochs (best_auc={best_auc:.4f}).")
    if best_state is not None and not OUT_CKPT.is_file():
        atomic_torch_save({
            "arch": "convnext_tiny",
            "classes": classes,
            "weapon_classes": ["weapon"],
            "img_size": IMG_SIZE,
            "mean": MEAN,
            "std": STD,
            "state_dict": best_state,
        }, OUT_CKPT)
else:
    ys, preds = [], []
    for epoch in range(start_epoch, EPOCHS + 1):
        model.train()
        tr_loss = 0.0
        for x, y in train_loader:
            x, y = x.cuda(non_blocking=True), y.cuda(non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda"):
                loss = criterion(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            tr_loss += loss.item() * x.size(0)
        scheduler.step()

        model.eval()
        ys, preds, wprobs = [], [], []
        with torch.no_grad():
            for x, y in val_loader:
                x = x.cuda(non_blocking=True)
                with torch.amp.autocast("cuda"):
                    probs = torch.softmax(model(x).float(), dim=-1)
                ys.extend(y.tolist())
                preds.extend(probs.argmax(dim=-1).cpu().tolist())
                wprobs.extend(probs[:, weapon_idx].cpu().tolist())

        bin_y = [1 if t == weapon_idx else 0 for t in ys]
        auc = roc_auc_score(bin_y, wprobs) if len(set(bin_y)) > 1 else 0.0
        acc = float(np.mean(np.array(ys) == np.array(preds)))
        print(f"Epoch {epoch:02d}/{EPOCHS} | loss={tr_loss/len(train_items):.4f} | val_acc={acc:.4f} | weapon_AUC={auc:.4f}")

        if auc > best_auc:
            best_auc = auc
            best_state = copy.deepcopy({k: v.cpu() for k, v in model.state_dict().items()})
            atomic_torch_save({
                "arch": "convnext_tiny",
                "classes": classes,
                "weapon_classes": ["weapon"],
                "img_size": IMG_SIZE,
                "mean": MEAN,
                "std": STD,
                "state_dict": best_state,
            }, OUT_CKPT)
            print(f"  [Saved Best -> Drive] {OUT_CKPT.name} (AUC={best_auc:.4f})")

        atomic_torch_save({
            "epoch": epoch,
            "classes": classes,
            "train_rel": train_rel,
            "val_rel": val_rel,
            "best_auc": best_auc,
            "best_state": best_state,
            "model_state": {k: v.cpu() for k, v in model.state_dict().items()},
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "scaler_state": scaler.state_dict(),
        }, RESUME_CKPT)

    print("\nBest validation weapon ROC-AUC:", round(best_auc, 4))
    if ys and preds:
        print(classification_report(ys, preds, target_names=classes, digits=4))

print("Ready checkpoint on Drive:", OUT_CKPT)

# Optional: trigger direct browser download of the ~109 MB checkpoint
# from google.colab import files
# files.download(str(OUT_CKPT))


## 3. Pipeline B (Heavy GPU): Stage-1 Multi-Class + Explicit Distractor Training (`YOLO11x` & `RT-DETR-L`)

Instead of merging everything into `0: weapon` with empty background labels:
- **Positive weapon sub-classes** keep distinct aspect-ratio priors (`0: handgun`, `1: long_gun`, `2: knife`), which can be unified at alert time.
- **Explicit distractor classes** (`3: phone`, `4: umbrella`) force the backbone features to cleanly separate handheld objects from real firearms.
- **`RT-DETR-L` comparison:** Uses global self-attention and **zero NMS**, which directly helps when rifles are stacked/piled tightly or near table edges.


In [ ]:
# Cell B1: Prepare Unified Multi-Class + Explicit Distractor Dataset YAML in /content/work
import getpass
import yaml
from pathlib import Path

MULTI_DIR = WORK_DIR / "weapon_multiclass_v2"
DATA_YAML = MULTI_DIR / "data.yaml"
weapon_loc = DL_DIR / "weapon_v5"
phone_loc = DL_DIR / "phone_v3"
umbrella_loc = DL_DIR / "umbrella_v1"

# Ensure raw YOLO splits exist even if Cell A2 loaded crops from Drive cache
if not ((weapon_loc / "data.yaml").is_file() and (phone_loc / "data.yaml").is_file() and (umbrella_loc / "data.yaml").is_file()):
    from roboflow import Roboflow
    ROBOFLOW_KEY = os.environ.get("ROBOFLOW_API_KEY") or getpass.getpass("Enter Roboflow API key for Stage-1 datasets: ")
    rf = Roboflow(api_key=ROBOFLOW_KEY)
    if not (weapon_loc / "data.yaml").is_file():
        rf.workspace("yolov7test-u13vc").project("weapon-detection-m7qso").version(5).download("yolov8", location=str(weapon_loc))
    if not (phone_loc / "data.yaml").is_file():
        rf.workspace("fyp-kbmf2").project("phone-finder-2.0-fbxkx").version(3).download("yolov8", location=str(phone_loc))
    if not (umbrella_loc / "data.yaml").is_file():
        rf.workspace("objectdetection-gd0bu").project("umbrella-6hxa2").version(1).download("yolov8", location=str(umbrella_loc))

for split in ("train", "valid"):
    (MULTI_DIR / split / "images").mkdir(parents=True, exist_ok=True)
    (MULTI_DIR / split / "labels").mkdir(parents=True, exist_ok=True)

# Target schema: 0: handgun | 1: long_gun | 2: knife | 3: phone | 4: umbrella
HANDGUN_KW = {"pistol", "handgun", "revolver", "curta", "arma_curta"}
LONGGUN_KW = {"rifle", "shotgun", "long", "larga", "arma_larga", "smg", "weapon", "gun"}
KNIFE_KW = {"knife", "blade", "machete", "dagger"}

def import_yolo_split(src_root: Path, forced_class_id: int | None = None, prefix: str = "ds"):
    meta = yaml.safe_load((src_root / "data.yaml").read_text()) if (src_root / "data.yaml").is_file() else {}
    names = meta.get("names", {})
    id_to_name = {int(k): str(v).lower() for k, v in (names.items() if isinstance(names, dict) else enumerate(names))}

    for src_split, dst_split in (("train", "train"), ("valid", "valid"), ("val", "valid")):
        if not (src_root / src_split / "images").is_dir():
            continue
        for img_p in (src_root / src_split / "images").iterdir():
            if img_p.suffix.lower() not in (".jpg", ".jpeg", ".png"):
                continue
            lbl_p = src_root / src_split / "labels" / f"{img_p.stem}.txt"
            new_lines = []
            if lbl_p.is_file():
                for line in lbl_p.read_text().strip().splitlines():
                    parts = line.split()
                    if len(parts) < 5:
                        continue
                    src_cid = int(float(parts[0]))
                    if forced_class_id is not None:
                        new_cid = forced_class_id
                    else:
                        cname = id_to_name.get(src_cid, "")
                        if any(k in cname for k in KNIFE_KW):
                            new_cid = 2
                        elif any(k in cname for k in HANDGUN_KW):
                            new_cid = 0
                        elif any(k in cname for k in LONGGUN_KW):
                            new_cid = 1
                        else:
                            continue  # skip non-weapon annotations like 'person' or 'hand'
                    new_lines.append(f"{new_cid} " + " ".join(parts[1:5]))

            dst_stem = f"{prefix}_{img_p.stem}"
            dst_img = MULTI_DIR / dst_split / "images" / f"{dst_stem}{img_p.suffix}"
            if not dst_img.exists():
                dst_img.symlink_to(img_p.resolve())
            (MULTI_DIR / dst_split / "labels" / f"{dst_stem}.txt").write_text("\n".join(new_lines))

import_yolo_split(weapon_loc, forced_class_id=None, prefix="wpn")
import_yolo_split(phone_loc, forced_class_id=3, prefix="phn")
import_yolo_split(umbrella_loc, forced_class_id=4, prefix="umb")

DATA_YAML.write_text(yaml.safe_dump({
    "path": str(MULTI_DIR),
    "train": "train/images",
    "val": "valid/images",
    "names": {0: "handgun", 1: "long_gun", 2: "knife", 3: "phone", 4: "umbrella"},
}, sort_keys=False))
print("Created multi-class + explicit distractor dataset YAML at:", DATA_YAML)


In [ ]:
# Cell B2: Crash-Proof YOLO11x Fine-Tuning (Auto-Resume from Drive every epoch)
import torch
import albumentations as A
from ultralytics import YOLO

FORCE_RESTART = False  # Set True only if you want to discard Drive checkpoints and start fresh
START_WEIGHTS = str(DRIVE_RUNS / "yolo11x_weapon_production.pt")
if not Path(START_WEIGHTS).is_file():
    START_WEIGHTS = "yolo11x.pt"

LAST_PT = DRIVE_RUNS / "yolo11x_multiclass_v2" / "weights" / "last.pt"
BEST_PT = DRIVE_RUNS / "yolo11x_multiclass_v2" / "weights" / "best.pt"

cctv_augments = [
    A.MotionBlur(blur_limit=(3, 7), p=0.20),
    A.ImageCompression(quality_range=(35, 85), p=0.35),
    A.ISONoise(p=0.20),
    A.ToGray(p=0.10),
]

should_start_fresh = FORCE_RESTART or not LAST_PT.is_file()
if not should_start_fresh:
    ckpt_meta = torch.load(LAST_PT, map_location="cpu", weights_only=False)
    last_epoch = int(ckpt_meta.get("epoch", -1))
    if last_epoch == -1:
        print(f"[Complete] YOLO11x training already finished! Best weights at: {BEST_PT}")
    else:
        print(f"[Auto-Resume] Resuming YOLO11x from epoch {last_epoch + 1} via {LAST_PT}")
        YOLO(str(LAST_PT)).train(resume=True)
else:
    model_y = YOLO(START_WEIGHTS)
    model_y.train(
        data=str(DATA_YAML),
        epochs=40,
        imgsz=800,
        batch=8,               # Safe for 16GB T4 (increase to 16 on L4/A100)
        nbs=64,                # Gradient accumulation matches batch=64 stability
        single_cls=False,      # CRITICAL: keep separate classes instead of inheriting single_cls=True
        optimizer="AdamW",     # CRITICAL: prevents 'auto' from overriding lr0
        lr0=1.5e-4,
        lrf=0.05,
        cos_lr=True,
        amp=True,
        save=True,
        save_period=-1,        # Overwrites last.pt every epoch without exhausting Drive quota
        augmentations=cctv_augments,
        project=str(DRIVE_RUNS),
        name="yolo11x_multiclass_v2",
        exist_ok=True,
    )


In [ ]:
# Cell B3: Zero-NMS Global Context RT-DETR-L Benchmark (Auto-Resume from Drive every epoch)
import torch
from ultralytics import RTDETR

FORCE_RESTART = False
LAST_RT = DRIVE_RUNS / "rtdetr_l_multiclass_v2" / "weights" / "last.pt"
BEST_RT = DRIVE_RUNS / "rtdetr_l_multiclass_v2" / "weights" / "best.pt"

should_start_fresh = FORCE_RESTART or not LAST_RT.is_file()
if not should_start_fresh:
    ckpt_meta = torch.load(LAST_RT, map_location="cpu", weights_only=False)
    last_epoch = int(ckpt_meta.get("epoch", -1))
    if last_epoch == -1:
        print(f"[Complete] RT-DETR-L training already finished! Best weights at: {BEST_RT}")
    else:
        print(f"[Auto-Resume] Resuming RT-DETR-L from epoch {last_epoch + 1} via {LAST_RT}")
        RTDETR(str(LAST_RT)).train(resume=True)
else:
    model_rtdetr = RTDETR("rtdetr-l.pt")
    model_rtdetr.train(
        data=str(DATA_YAML),
        epochs=40,
        imgsz=640,             # Use 800 on Colab Pro L4/A100
        batch=8,
        optimizer="AdamW",     # CRITICAL: prevents 'auto' from forcing lr=0.001429 (NaN blowup)
        lr0=1e-4,
        amp=True,
        save=True,
        save_period=-1,        # Overwrites last.pt every epoch without exhausting Drive quota
        project=str(DRIVE_RUNS),
        name="rtdetr_l_multiclass_v2",
        exist_ok=True,
    )


## 4. Golden Stress-Test Benchmark: FAPH (False Alarms Per Hour) & Incident Recall / TTFD

Validation mAP misses the real production failure modes. Place two small folders in Drive (`/content/drive/MyDrive/weapon_v2_runs/golden_test/`):
- `negative_clips/` — weapon-free videos of phones at oblique angles, umbrellas, folding tables, empty corridors
- `positive_clips/` — real weapon videos (rapid transitions, occluded grips, CCTV angles)

This cell measures **FAPH** (False Alarms Per Hour), **Incident Recall**, and **Time-To-First-Detection (TTFD)** comparing Raw Stage-1 vs. Stage-1 + Stage-2 Verifier.


In [ ]:
import time
import cv2
import numpy as np
import torch
import torchvision.models as tv_models
from pathlib import Path
from ultralytics import YOLO

GOLDEN_DIR = DRIVE_RUNS / "golden_test"
NEG_DIR = GOLDEN_DIR / "negative_clips"
POS_DIR = GOLDEN_DIR / "positive_clips"

STAGE1_PT = DRIVE_RUNS / "yolo11x_multiclass_v2/weights/best.pt"
if not STAGE1_PT.is_file():
    STAGE1_PT = DRIVE_RUNS / "yolo11x_weapon_production.pt"
VERIFIER_PT = DRIVE_RUNS / "crop_verifier_convnext_tiny.pt"

if not NEG_DIR.is_dir() and not POS_DIR.is_dir():
    print(f"Create {NEG_DIR} and {POS_DIR} with your test .mp4 clips and re-run this cell.")
else:
    det_model = YOLO(str(STAGE1_PT))
    v_ckpt = torch.load(VERIFIER_PT, map_location="cpu", weights_only=True) if VERIFIER_PT.is_file() else None
    v_model = None
    if v_ckpt is not None:
        v_model = tv_models.convnext_tiny(weights=None)
        v_model.classifier[2] = torch.nn.Linear(v_model.classifier[2].in_features, len(v_ckpt["classes"]))
        v_model.load_state_dict(v_ckpt["state_dict"])
        v_model = v_model.cuda().eval()
        v_widx = v_ckpt["classes"].index("weapon")
        v_mean = torch.tensor(v_ckpt["mean"], device="cuda").view(1, 3, 1, 1)
        v_std = torch.tensor(v_ckpt["std"], device="cuda").view(1, 3, 1, 1)

    def eval_folder(folder: Path, is_positive: bool):
        clips = sorted([p for p in folder.glob("*") if p.suffix.lower() in (".mp4", ".avi", ".mov", ".mkv")])
        total_sec = 0.0
        raw_fp_frames, ver_fp_frames = 0, 0
        raw_hits, ver_hits = 0, 0
        raw_ttfd, ver_ttfd = [], []

        for clip in clips:
            cap = cv2.VideoCapture(str(clip))
            fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
            fidx = 0
            clip_raw_t, clip_ver_t = None, None
            while True:
                ret, frame = cap.read()
                if not ret:
                    break
                res = det_model.predict(frame, imgsz=800, conf=0.35, verbose=False)[0]
                weapon_boxes = []
                for b in res.boxes:
                    cname = str(det_model.names[int(b.cls[0])]).lower()
                    if cname in ("phone", "umbrella"):
                        continue
                    weapon_boxes.append((list(map(int, b.xyxy[0])), float(b.conf[0])))

                raw_hit = len(weapon_boxes) > 0
                ver_hit = False
                if raw_hit:
                    if v_model is None:
                        ver_hit = True
                    else:
                        for (x1, y1, x2, y2), conf in weapon_boxes:
                            if conf >= 0.85:
                                ver_hit = True
                                break
                            bw, bh = max(1, x2 - x1), max(1, y2 - y1)
                            fh, fw = frame.shape[:2]
                            px1, py1 = max(0, x1 - int(0.3 * bw)), max(0, y1 - int(0.3 * bh))
                            px2, py2 = min(fw, x2 + int(0.3 * bw)), min(fh, y2 + int(0.3 * bh))
                            crop = cv2.cvtColor(cv2.resize(frame[py1:py2, px1:px2], (224, 224)), cv2.COLOR_BGR2RGB)
                            t = (torch.from_numpy(crop).permute(2, 0, 1).unsqueeze(0).cuda().float() / 255.0 - v_mean) / v_std
                            with torch.no_grad():
                                p_w = torch.softmax(v_model(t), dim=-1)[0, v_widx].item()
                            if p_w >= 0.50:
                                ver_hit = True
                                break

                if raw_hit:
                    raw_fp_frames += 1
                    if clip_raw_t is None:
                        clip_raw_t = fidx / fps
                if ver_hit:
                    ver_fp_frames += 1
                    if clip_ver_t is None:
                        clip_ver_t = fidx / fps
                fidx += 1
            cap.release()
            total_sec += fidx / fps
            if clip_raw_t is not None:
                raw_hits += 1
                raw_ttfd.append(clip_raw_t)
            if clip_ver_t is not None:
                ver_hits += 1
                ver_ttfd.append(clip_ver_t)

        if not clips:
            return
        if is_positive:
            print(f"[Positive Clips={len(clips)}] Raw Recall={raw_hits}/{len(clips)} (TTFD={np.mean(raw_ttfd or [0]):.2f}s) | +Verifier Recall={ver_hits}/{len(clips)} (TTFD={np.mean(ver_ttfd or [0]):.2f}s)")
        else:
            hrs = max(total_sec / 3600.0, 1e-6)
            print(f"[Negative Clips={len(clips)}, {total_sec:.1f}s] Raw FP frames={raw_fp_frames} ({raw_fp_frames/hrs:.1f}/hr) | +Verifier FP frames={ver_fp_frames} ({ver_fp_frames/hrs:.1f}/hr)")

    eval_folder(NEG_DIR, is_positive=False)
    eval_folder(POS_DIR, is_positive=True)
